In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

from src.arranger.arranger import arrange_chord_sequence, humanizar_eventos_midi
from src.synth.midi_renderer import *
from src.synth.midi_renderer import synthesize_with_drums
from src.audio.pitch import *
from src.audio.chroma import extract_chroma
from src.harmony.viterbi import suggest_chords_viterbi
from IPython.display import Audio
import librosa
import pretty_midi

## Teste do arranjador

In [2]:
audio_path = '../data/audio-segment1-acima-do-sol-lead.mp3'
TOM='G'
BPM=92
OFFSET=1
UPC = 2
y, sr = cut_vocal(audio_path, BPM, 0, 8)
listen_beats(y, sr, BPM, offset=OFFSET)
time, frequency, confidence = extract_f0_pyin(y, sr)
df_compassos = extract_chroma(
    time,
    frequency,
    confidence,
    BPM,
    tom=TOM,
    offset=OFFSET,
    unidades_por_compasso=UPC,
    quantizar_para_tom=True
)
print(df_compassos)

acordes, caminho, funcoes, seg = suggest_chords_viterbi(
  df_compassos, 
  tom_raiz=TOM, 
  tamanho_bloco=1,
  custo_troca_fora_tempo_forte=0.6, 
  permitir_diminutos=False,
)

df_compassos['acorde'] = acordes 
df_compassos['funcao'] = df_compassos['acorde'].map(funcoes)

print(df_compassos[['compasso', 'nota_dominante', 'acorde', 'funcao']])


c:\Users\Cliente\Documentos\Projeto_de_pesquisa\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    unidade  compasso  posicao_no_compasso  tempo_inicio  tempo_fim  \
0         1         1                    0          0.65       1.96   
1         2         1                    1          1.96       3.26   
2         3         2                    0          3.26       4.57   
3         4         2                    1          4.57       5.87   
4         5         3                    0          5.87       7.17   
5         6         3                    1          7.17       8.48   
6         7         4                    0          8.48       9.78   
7         8         4                    1          9.78      11.09   
8         9         5                    0         11.09      12.39   
9        10         5                    1         12.39      13.70   
10       11         6                    0         13.70      15.00   
11       12         6                    1         15.00      16.30   
12       13         7                    0         16.30      17.61   
13    

In [3]:
atividade_vocal = extract_vocal_activity(
    time,
    frequency,
    confidence,
    BPM,
    OFFSET,
    unidades_por_compasso=UPC,
    min_voiced_ratio=0.3
)

print(atividade_vocal)

[np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_]


In [4]:
acordes_sugeridos = acordes if UPC == 1 else seg
eventos_midi = arrange_chord_sequence(
    acordes_sugeridos=acordes_sugeridos,
    tom_raiz=TOM,
    atividade_vocal=atividade_vocal,
    bpm=BPM,
    unidades_por_compasso_vocal=UPC,
    offset=OFFSET
)

pm = export_events_to_midi(eventos_midi, '../data/midis/Acima_do_sol.mid')
audio = synthesize_with_drums(pm, drum_gain=256.0)
## audio = pm.synthesize(fs=22050)
offset = int(0 * (60 / BPM) * 22050)

# 2. Aplica o pad no áudio sintetizado
audio_final = np.pad(audio, (offset, 0), mode="constant")

# 3. Igualar os tamanhos de audio_final e y para permitir a soma sem erro de broadcast
tamanho_max = max(len(audio_final), len(y))
audio_final_aligned = np.pad(audio_final, (0, tamanho_max - len(audio_final)))
y_aligned = np.pad(y, (0, tamanho_max - len(y)))

# 4. Mixagem final e reprodução
audio_mix = audio_final_aligned + 512.0 * y_aligned
Audio(audio_mix, rate=22050)

## Humanizando mais o MIDI

In [5]:
eventos_humanizados = humanizar_eventos_midi(eventos_midi)

pm = export_events_to_midi(eventos_humanizados, "../data/midis/Acima_do_Sol2.mid")
audio = synthesize_with_drums(pm, drum_gain=0.0)
offset = int(0 * (60 / BPM) * 22050)
audio_final = np.pad(audio, (offset, 0), mode='constant')
tamanho_max = max(len(audio_final), len(y))
audio_final_aligned = np.pad(audio_final, (0, tamanho_max - len(audio_final)))
y_aligned = np.pad(y, (0, tamanho_max - len(y)))

audio_mix = audio_final_aligned + 512.0 * y_aligned
Audio(audio_mix, rate=22050)

## Renderizando com Soundfonts

In [8]:
SOUNDFONT_PATH = '../data/soundfont/Yamaha_C7_Normalized.sf2'
audio_wav = render_audio_wav(pm, caminho_wav='../data/wavs/Acima_do_Sol2.wav')

Audio(audio_wav, rate=44100)